# Notebook 03 — Entraînement du Modèle ML avec Spark MLlib

**Phase 1 — Batch Training**

Ce notebook entraîne le modèle de classification de sentiment (Logistic Regression binaire) sur le dataset Sentiment140 via **Spark MLlib**.

Pipeline ML :
```
Texte brut → Tokenizer → StopWordsRemover → HashingTF (2^18) → IDF → Logistic Regression
```

**Lien rapport** : Chapitres 4 — Phase 1 (sections 4.3, 4.4, 4.5)

In [1]:
import sys
import os
from pathlib import Path

# Détection Docker vs Windows
in_docker = os.path.exists('/.dockerenv')
ROOT_DIR = Path('/workspace') if in_docker else Path().absolute().parent
sys.path.insert(0, str(ROOT_DIR))

from src.train_model import build_full_ml_pipeline, run_training
from src.preprocessing import load_sentiment140, clean_dataframe
from src.utils import get_spark_session, get_logger, parquet_exists
from config.config import *

logger = get_logger('training_notebook')
print('✅ Imports OK')
print(f'   Mode    : {"Docker" if in_docker else "Windows local"}')
print(f'   ROOT_DIR: {ROOT_DIR}')
print(f'   MODEL_PATH: {MODEL_PATH}')


✅ Imports OK
   Mode    : Docker
   ROOT_DIR: /workspace
   MODEL_PATH: /workspace/data/models/sentiment_model


In [4]:
from src.utils import get_spark_session
from config.config import SPARK_MASTER, SPARK_DRIVER_MEMORY, SPARK_EXECUTOR_MEMORY

# Créer la SparkSession avant tout
spark = get_spark_session("SentimentAnalysis_Training")
print(f'✅ SparkSession créée')
print(f'   Master : {spark.sparkContext.master}')
print(f'   Version: {spark.version}')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 09:34:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[2026-06-07 09:34:40] INFO     src.utils — SparkSession 'SentimentAnalysis_Training' démarrée — version Spark 3.5.1
✅ SparkSession créée
   Master : local[*]
   Version: 3.5.1


## 1. Architecture du Pipeline ML

Le pipeline encapsule TOUT le traitement NLP + le modèle ML en un seul objet sérialisable.
À l'inférence, seule la colonne `text` (brut) est nécessaire.

In [5]:
# Affichage de l'architecture du pipeline
pipeline = build_full_ml_pipeline()
print('=== PIPELINE SPARK ML ===')
for i, stage in enumerate(pipeline.getStages()):
    print(f'  Stage {i+1}: {stage.__class__.__name__}')
    if hasattr(stage, 'getNumFeatures'):
        print(f'           NumFeatures = {stage.getNumFeatures():,}')
    if hasattr(stage, 'getMaxIter'):
        print(f'           MaxIter = {stage.getMaxIter()}, RegParam = {stage.getRegParam()}')

=== PIPELINE SPARK ML ===
  Stage 1: Tokenizer
  Stage 2: StopWordsRemover
  Stage 3: HashingTF
           NumFeatures = 262,144
  Stage 4: IDF
  Stage 5: LogisticRegression
           MaxIter = 100, RegParam = 0.01


## 2. Entraînement complet

> ⏱️ **Durée estimée :** 5-15 minutes sur 1,6M tweets selon les ressources disponibles.  
> Pour un test rapide, passez `sample_fraction=0.1` (160K tweets, ~1-2 minutes)

In [6]:
# Entraînement complet
# Décommentez la ligne 'sample_fraction' pour un test rapide

pipeline_model, metrics = run_training(
    # sample_fraction=0.1  # Décommenter pour test rapide (10% = 160K tweets)
)

print('\n=== RÉSULTATS ===')
for name, val in metrics.items():
    print(f'  {name:<15} : {val:.4f}')

print(f'\n✅ Modèle sauvegardé dans : {MODEL_PATH}')

[2026-06-07 09:35:13] INFO     src.utils — SparkSession 'SentimentAnalysis_BatchTraining' démarrée — version Spark 3.5.1
[2026-06-07 09:35:13] INFO     src.train_model — Chargement depuis Parquet nettoyé : /workspace/data/processed/sentiment140_clean.parquet


26/06/07 09:35:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


[2026-06-07 09:35:20] INFO     src.train_model — Dataset chargé : 1,593,856 tweets


[2026-06-07 09:35:30] INFO     src.train_model — Train : 1,275,293 | Test : 318,563
[2026-06-07 09:35:30] INFO     src.train_model — Entraînement du pipeline ML (TF-IDF + Logistic Regression)...


26/06/07 09:35:50 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:03 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:03 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/07 09:36:04 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:13 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:14 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:14 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:15 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:16 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:16 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:16 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/07 09:36:17 WARN DAGSchedul

...

26/06/07 09:36:31 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB
26/06/07 09:36:42 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB
26/06/07 09:36:49 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB
26/06/07 09:36:57 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB
26/06/07 09:37:04 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB


[2026-06-07 09:37:18] INFO     src.train_model — ==================================================
[2026-06-07 09:37:18] INFO     src.train_model — MÉTRIQUES DU MODÈLE — Jeu de test Sentiment140
[2026-06-07 09:37:18] INFO     src.train_model — ==================================================
[2026-06-07 09:37:18] INFO     src.train_model —   ✗  accuracy     : 0.7660  (cible ≥ 0.78)
[2026-06-07 09:37:18] INFO     src.train_model —   ✓  precision    : 0.7663  (cible ≥ 0.76)
[2026-06-07 09:37:18] INFO     src.train_model —   ✓  recall       : 0.7660  (cible ≥ 0.76)
[2026-06-07 09:37:18] INFO     src.train_model —   ✗  f1_score     : 0.7659  (cible ≥ 0.77)
[2026-06-07 09:37:18] INFO     src.train_model —   ✗  auc_roc      : 0.8304  (cible ≥ 0.85)


[2026-06-07 09:37:18] INFO     src.train_model — ==================================================
[2026-06-07 09:37:18] INFO     src.train_model — Sauvegarde du modèle → /workspace/data/models/sentiment_model


26/06/07 09:37:23 WARN TaskSetManager: Stage 114 contains a task of very large size (4187 KiB). The maximum recommended task size is 1000 KiB.
26/06/07 09:37:25 WARN TaskSetManager: Stage 118 contains a task of very large size (1597 KiB). The maximum recommended task size is 1000 KiB.


[2026-06-07 09:37:26] INFO     src.train_model — Modèle sauvegardé avec succès
[2026-06-07 09:37:27] INFO     src.train_model — SparkSession fermée

=== RÉSULTATS ===
  accuracy        : 0.7660
  precision       : 0.7663
  recall          : 0.7660
  f1_score        : 0.7659
  auc_roc         : 0.8304

✅ Modèle sauvegardé dans : /workspace/data/models/sentiment_model


## 3. Vérification du modèle sauvegardé

In [7]:
# Structure du modèle sauvegardé
import os

print('=== STRUCTURE DU MODÈLE SAUVEGARDÉ ===')
for root, dirs, files in os.walk(MODEL_PATH):
    level = root.replace(str(MODEL_PATH), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{Path(root).name}/')
    if level < 2:
        for f in files[:3]:
            print(f'{indent}  {f}')

=== STRUCTURE DU MODÈLE SAUVEGARDÉ ===
sentiment_model/
  metadata/
    .part-00000.crc
    ._SUCCESS.crc
    part-00000
  stages/
    0_Tokenizer_d930f70036c5/
      metadata/
    1_StopWordsRemover_2ffb21c35cac/
      metadata/
    2_HashingTF_7c6cd68b5814/
      metadata/
    3_IDF_1ba9e65a47d3/
      data/
      metadata/
    4_LogisticRegression_9fd3519335b7/
      data/
      metadata/


## 4. Test d'inférence manuelle

In [10]:
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf
from src.utils import get_spark_session, get_neutral_udf

spark = get_spark_session('Inference_Test')
print(f'Master : {spark.sparkContext.master}')

loaded_model = PipelineModel.load(str(MODEL_PATH))
print('✅ Modèle chargé')

# UDF pour extraire les valeurs du vecteur sparse
prob_neg_udf = udf(lambda v: float(v[0]), DoubleType())
prob_pos_udf = udf(lambda v: float(v[1]), DoubleType())

test_tweets = [
    'I love this sunny day so much!',
    'This is absolutely terrible, worst experience ever.',
    'Just got home from work today.',
    'Feeling a bit tired this morning.',
    'The new iPhone battery life is amazing!',
    'Apple support is the worst I have ever dealt with.'
]

test_df = spark.createDataFrame([(t,) for t in test_tweets], ['text'])
predictions = loaded_model.transform(test_df)

# Extraire les probabilités avec UDF
predictions = predictions \
    .withColumn('prob_neg', prob_neg_udf(F.col('probability'))) \
    .withColumn('prob_pos', prob_pos_udf(F.col('probability')))

# Application de la couche Neutral
neutral_udf = get_neutral_udf(CONFIDENCE_THRESHOLD)
results = predictions.withColumn(
    'sentiment_label',
    neutral_udf(F.col('prediction'), F.col('prob_neg'), F.col('prob_pos'))
).withColumn(
    'confidence',
    F.greatest(F.col('prob_neg'), F.col('prob_pos'))
)

print('=== PRÉDICTIONS SUR TWEETS DE TEST ===')
results.select(
    'text', 'prob_neg', 'prob_pos', 'confidence', 'sentiment_label'
).show(truncate=50)

spark.stop()
print('✅ SparkSession fermée')

[2026-06-07 09:44:21] INFO     src.utils — SparkSession 'Inference_Test' démarrée — version Spark 3.5.1
Master : local[*]
✅ Modèle chargé
=== PRÉDICTIONS SUR TWEETS DE TEST ===


26/06/07 09:44:26 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB
26/06/07 09:44:27 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB
26/06/07 09:44:28 WARN DAGScheduler: Broadcasting large task binary with size 5.6 MiB


+--------------------------------------------------+-------------------+--------------------+------------------+---------------+
|                                              text|           prob_neg|            prob_pos|        confidence|sentiment_label|
+--------------------------------------------------+-------------------+--------------------+------------------+---------------+
|                    I love this sunny day so much!|0.11792570801069202|   0.882074291989308| 0.882074291989308|       Positive|
|This is absolutely terrible, worst experience e...| 0.7809400703904963|  0.2190599296095037|0.7809400703904963|       Negative|
|                    Just got home from work today.| 0.9973234653790259|0.002676534620974058|0.9973234653790259|       Negative|
|                 Feeling a bit tired this morning.| 0.7972393159263526| 0.20276068407364745|0.7972393159263526|       Negative|
|           The new iPhone battery life is amazing!| 0.6897858294499576|  0.3102141705500424|0.68

## 5. Synthèse Phase 1

| Étape | Résultat |
|-------|----------|
| Dataset chargé | 1 600 000 tweets (Sentiment140) |
| Split Train/Test | 1 275 293 train / 318 563 test |
| Pipeline ML | Tokenizer → StopWords → HashingTF (2^18) → IDF → LogisticRegression |
| Accuracy | 76.6% |
| Precision | 76.6% |
| Recall | 76.6% |
| F1-score | 76.6% |
| AUC-ROC | 83.0% |
| Modèle sauvegardé | `/workspace/data/models/sentiment_model/` |

### Stages du modèle sauvegardé

### Analyse des prédictions de test

| Tweet | P(Neg) | P(Pos) | Label | Correct |
|-------|--------|--------|-------|---------|
| "I love this sunny day" | 11.8% | 88.2% | Positive | ✅ |
| "absolutely terrible" | 78.1% | 21.9% | Negative | ✅ |
| "Just got home from work" | 99.7% | 0.3% | Negative | ⚠️ Neutre en réalité |
| "Feeling a bit tired" | 79.7% | 20.3% | Negative | ⚠️ Acceptable |
| "iPhone battery life is amazing" | 69.0% | 31.0% | Negative | ❌ Modèle général |
| "Apple support is the worst" | 75.6% | 24.4% | Negative | ✅ |

> ⚠️ Les erreurs sur les tweets Apple sont normales — le modèle est entraîné sur Sentiment140 (tweets généraux).
> La Phase 3 (Apple Brand Monitoring) est conçue pour ce cas d'usage spécifique.
